In [1]:
import cv2
from ultralytics import YOLO

def main():
    model = YOLO('yolov8n.pt')

    video_path = 'E:\Computer Vision\LAB\lab10\highway (1).mp4'
    cap = cv2.VideoCapture(video_path)

    print("Starting video processing... Press 'q' to exit.")

    while cap.isOpened():
        ret, frame = cap.read()
        
        if not ret:
            print("Video stream ended or cannot be read.")
            break

        results = model(frame, stream=True)
        for r in results:
            annotated_frame = r.plot()
            cv2.imshow("YOLOv8 Object Detection", annotated_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Starting video processing... Press 'q' to exit.

0: 384x640 1 car, 215.0ms
Speed: 14.0ms preprocess, 215.0ms inference, 22.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 102.8ms
Speed: 16.3ms preprocess, 102.8ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 134.6ms
Speed: 2.6ms preprocess, 134.6ms inference, 3.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 train, 97.5ms
Speed: 3.5ms preprocess, 97.5ms inference, 2.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 104.9ms
Speed: 3.9ms preprocess, 104.9ms inference, 2.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 116.5ms
Speed: 2.6ms preprocess, 116.5ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 108.5ms
Speed: 2.9ms preprocess, 108.5ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 109.9ms
Speed: 2.9ms preprocess, 109.9ms inferen

In [3]:
import cv2
import math
from ultralytics import YOLO

class CentroidTracker:
    def __init__(self, max_distance=50):
        # A dictionary is used to store object IDs and their previous positions
        self.objects = {} 
        self.next_id = 0
        # Maximum Euclidean distance to consider a detection the same object
        self.max_distance = max_distance

    def update(self, detections):
        # If no objects are currently tracked, assign new IDs to all detections
        if len(self.objects) == 0:
            for cx, cy, box in detections:
                self.objects[self.next_id] = (cx, cy, box)
                self.next_id += 1
            return self.objects

        updated_objects = {}
        used_detections = set()

        # Match new detections to existing tracked objects
        for obj_id, (prev_cx, prev_cy, _) in self.objects.items():
            best_match_idx = -1
            min_dist = float('inf')

            for i, (cx, cy, _) in enumerate(detections):
                if i in used_detections:
                    continue
                
                # New detections are matched with existing objects using Euclidean distance
                dist = math.hypot(cx - prev_cx, cy - prev_cy)
                
                if dist < min_dist and dist < self.max_distance:
                    min_dist = dist
                    best_match_idx = i
            
            # If a match is found, the object retains its ID
            if best_match_idx != -1:
                cx, cy, box = detections[best_match_idx]
                updated_objects[obj_id] = (cx, cy, box)
                used_detections.add(best_match_idx)

        # Otherwise, a new ID is assigned for unmatched detections
        for i, (cx, cy, box) in enumerate(detections):
            if i not in used_detections:
                updated_objects[self.next_id] = (cx, cy, box)
                self.next_id += 1

        # Update the dictionary for the next frame
        self.objects = updated_objects
        return self.objects

def main():
    model = YOLO('yolov8n.pt')
    tracker = CentroidTracker(max_distance=50)

    video_path = 'E:\Computer Vision\LAB\lab10\highway (1).mp4' 
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: Could not open video.")
        return

    print("Starting video processing... Press 'q' to exit.")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, stream=True)
        
        detections = []
        for r in results:
            for box in r.boxes:
                # Extract bounding box coordinates
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                
                # The centroid of each bounding box is calculated
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                
                # Store the centroid and the bounding box 
                detections.append((cx, cy, (x1, y1, x2, y2)))

        # Update tracker with current frame's detections
        tracked_objects = tracker.update(detections)

        # Display unique IDs for each object
        for obj_id, (cx, cy, (x1, y1, x2, y2)) in tracked_objects.items():
            # Draw the bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
            
            # Draw the centroid
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
            
            # Display the ID text
            text = f"ID: {obj_id}"
            cv2.putText(frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        cv2.imshow("Centroid-based Tracking", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Starting video processing... Press 'q' to exit.

0: 384x640 1 car, 71.2ms
Speed: 2.9ms preprocess, 71.2ms inference, 1.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 112.0ms
Speed: 2.5ms preprocess, 112.0ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 108.6ms
Speed: 2.3ms preprocess, 108.6ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 1 train, 96.3ms
Speed: 2.4ms preprocess, 96.3ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 84.4ms
Speed: 2.4ms preprocess, 84.4ms inference, 3.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 76.2ms
Speed: 2.8ms preprocess, 76.2ms inference, 1.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 75.0ms
Speed: 2.4ms preprocess, 75.0ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 80.3ms
Speed: 2.4ms preprocess, 80.3ms inference, 1.1ms pos

In [4]:
import cv2
import math
from ultralytics import YOLO

class CentroidTracker:
    def __init__(self, max_distance=50):
        self.objects = {}
        self.next_id = 0
        self.max_distance = max_distance

    def update(self, detections):
        if len(self.objects) == 0:
            for cx, cy, box in detections:
                self.objects[self.next_id] = (cx, cy, box)
                self.next_id += 1
            return self.objects

        updated_objects = {}
        used_detections = set()

        for obj_id, (prev_cx, prev_cy, _) in self.objects.items():
            best_match_idx = -1
            min_dist = float('inf')

            for i, (cx, cy, _) in enumerate(detections):
                if i in used_detections:
                    continue
                
                dist = math.hypot(cx - prev_cx, cy - prev_cy)
                
                if dist < min_dist and dist < self.max_distance:
                    min_dist = dist
                    best_match_idx = i
            
            if best_match_idx != -1:
                cx, cy, box = detections[best_match_idx]
                updated_objects[obj_id] = (cx, cy, box)
                used_detections.add(best_match_idx)

        for i, (cx, cy, box) in enumerate(detections):
            if i not in used_detections:
                updated_objects[self.next_id] = (cx, cy, box)
                self.next_id += 1

        self.objects = updated_objects
        return self.objects


def main():
    model = YOLO('yolov8n.pt')
    tracker = CentroidTracker(max_distance=50)

    video_path = r'E:\Computer Vision\LAB\lab12\2165-155327596.mp4' 
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error: Could not open video.")
        return
    counted_ids = set()
    total_count = 0
    previous_y_positions = {}
    
    line_y = 350 

    print("Starting video processing... Press 'q' to exit.")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        cv2.line(frame, (0, line_y), (frame.shape[1], line_y), (0, 255, 255), 2)

        results = model(frame, stream=True)
        
        detections = []
        for r in results:
            for box in r.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                detections.append((cx, cy, (x1, y1, x2, y2)))

        tracked_objects = tracker.update(detections)

        # The output frame displays bounding boxes, object IDs, and the total count[cite: 31].
        for obj_id, (cx, cy, (x1, y1, x2, y2)) in tracked_objects.items():
            
            # --- Task 3 Additions: Line Crossing Logic ---
            # Check if we have a previous position for this object
            if obj_id in previous_y_positions:
                prev_cy = previous_y_positions[obj_id]
                
                # Check if the object crossed the line (top to bottom)
                if prev_cy < line_y and cy >= line_y:
                    # A record of counted IDs ensures that each object is counted only once[cite: 20].
                    if obj_id not in counted_ids:
                        total_count += 1
                        counted_ids.add(obj_id)
            previous_y_positions[obj_id] = cy
            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
            
            color = (0, 255, 0) if obj_id in counted_ids else (255, 0, 0)
            cv2.putText(frame, f"ID: {obj_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        cv2.putText(frame, f"Total Count: {total_count}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 3)

        cv2.imshow("Tracking & Counting", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Starting video processing... Press 'q' to exit.

0: 384x640 11 cars, 182.8ms
Speed: 203.9ms preprocess, 182.8ms inference, 6.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 9 cars, 162.1ms
Speed: 4.7ms preprocess, 162.1ms inference, 1.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 151.7ms
Speed: 2.9ms preprocess, 151.7ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 cars, 147.2ms
Speed: 3.0ms preprocess, 147.2ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 10 cars, 133.8ms
Speed: 3.7ms preprocess, 133.8ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 cars, 121.6ms
Speed: 3.0ms preprocess, 121.6ms inference, 10.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 11 cars, 140.2ms
Speed: 4.8ms preprocess, 140.2ms inference, 4.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 12 cars, 130.1ms
Speed: 3.0ms preprocess, 130.1ms

In [5]:
import cv2
import math
import time
from ultralytics import YOLO

class CentroidTracker:
    def __init__(self, max_distance=50):
        self.objects = {}
        self.next_id = 0
        self.max_distance = max_distance

    def update(self, detections):
        if len(self.objects) == 0:
            for cx, cy, box in detections:
                self.objects[self.next_id] = (cx, cy, box)
                self.next_id += 1
            return self.objects

        updated_objects = {}
        used_detections = set()

        for obj_id, (prev_cx, prev_cy, _) in self.objects.items():
            best_match_idx = -1
            min_dist = float('inf')

            for i, (cx, cy, _) in enumerate(detections):
                if i in used_detections:
                    continue
                
                dist = math.hypot(cx - prev_cx, cy - prev_cy)
                
                if dist < min_dist and dist < self.max_distance:
                    min_dist = dist
                    best_match_idx = i
            
            if best_match_idx != -1:
                cx, cy, box = detections[best_match_idx]
                updated_objects[obj_id] = (cx, cy, box)
                used_detections.add(best_match_idx)

        for i, (cx, cy, box) in enumerate(detections):
            if i not in used_detections:
                updated_objects[self.next_id] = (cx, cy, box)
                self.next_id += 1

        self.objects = updated_objects
        return self.objects


def main():
    model = YOLO('yolov8n.pt')
    tracker = CentroidTracker(max_distance=50)

    # --- Task 4 Addition: Use Webcam ---
    # 0 is typically the default integrated webcam. 
    # Use 1 or 2 if you are using an external USB camera.
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    counted_ids = set()
    total_count = 0
    previous_y_positions = {}
    
    # Adjust this value based on your webcam's resolution (e.g., 240 for a 480p feed)
    line_y = 240 

    print("Starting live webcam processing... Press 'q' to exit.")

    # Variables for overall performance evaluation
    frame_count = 0
    start_time_total = time.time()

    while cap.isOpened():
        # Start timer for real-time FPS calculation
        start_time = time.time()
        
        ret, frame = cap.read()
        if not ret:
            print("Failed to grab frame from webcam.")
            break
            
        cv2.line(frame, (0, line_y), (frame.shape[1], line_y), (0, 255, 255), 2)

        # verbose=False stops YOLO from printing detection details to the console every frame
        results = model(frame, stream=True, verbose=False)
        
        detections = []
        for r in results:
            for box in r.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cx = (x1 + x2) // 2
                cy = (y1 + y2) // 2
                detections.append((cx, cy, (x1, y1, x2, y2)))

        tracked_objects = tracker.update(detections)

        for obj_id, (cx, cy, (x1, y1, x2, y2)) in tracked_objects.items():
            if obj_id in previous_y_positions:
                prev_cy = previous_y_positions[obj_id]
                
                if prev_cy < line_y and cy >= line_y:
                    if obj_id not in counted_ids:
                        total_count += 1
                        counted_ids.add(obj_id)
            
            previous_y_positions[obj_id] = cy

            cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.circle(frame, (cx, cy), 4, (0, 0, 255), -1)
            
            color = (0, 255, 0) if obj_id in counted_ids else (255, 0, 0)
            cv2.putText(frame, f"ID: {obj_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        # --- Task 4 Addition: Performance Evaluation ---
        # Calculate instantaneous FPS
        end_time = time.time()
        fps = 1.0 / (end_time - start_time)
        frame_count += 1
        
        # Display Total Count and FPS on the feed
        cv2.putText(frame, f"Total Count: {total_count}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 255), 2)
        cv2.putText(frame, f"FPS: {int(fps)}", (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
        # -----------------------------------------------

        cv2.imshow("Real-Time Tracking & Counting", frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Calculate overall performance metrics upon exit
    total_time = time.time() - start_time_total
    avg_fps = frame_count / total_time if total_time > 0 else 0
    
    print("\n--- Performance Evaluation ---")
    print(f"Total Frames Processed: {frame_count}")
    print(f"Total Processing Time: {total_time:.2f} seconds")
    print(f"Average FPS: {avg_fps:.2f}")

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Starting live webcam processing... Press 'q' to exit.

--- Performance Evaluation ---
Total Frames Processed: 419
Total Processing Time: 43.06 seconds
Average FPS: 9.73
